# Frozen Rotation-Invariant Cross-Day Evaluation

This notebook loads the validation-selected adapter trained on
`New_Gesture_Trial_Dataset_Labeled (2).pt` and evaluates it without any weight
updates on `Gesture_Trial_Dataset_Labeled.pt`.

Ground-truth labels are used only for offline active-centered window extraction,
active-bin selection, and scoring. This is a cross-day classification test, not
the label-blind continuous online detector. Chance accuracy is 20% for five
gesture classes. Model hashes are checked before and after inference.

In [1]:
# ============================================================
# CELL 1 — CONFIGURATION + FROZEN META MODEL + 2 kHz ADAPTER
# ============================================================

from pathlib import Path
import sys
import random
import copy

import numpy as np
import torch
from torch import nn

try:
    from scipy.signal import butter, sosfiltfilt, resample_poly
except ImportError as exc:
    raise ImportError(
        "This notebook requires scipy for anti-aliased resampling and "
        "40 Hz high-pass filtering. Install it with: pip install scipy"
    ) from exc

try:
    from torch.nn.utils.parametrizations import weight_norm
except ImportError:
    from torch.nn.utils import weight_norm


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

REPO_ROOT = Path(r"C:\Users\Micah\utah-neuro\generic_neuromotor_interface")
MODEL_DIR = REPO_ROOT / "emg_models" / "discrete_gestures"
CKPT_PATH = MODEL_DIR / "model_checkpoint.ckpt"

BASE_DIR = REPO_ROOT
DATA_PATH = BASE_DIR / "New_Gesture_Trial_Dataset_Labeled (2).pt"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from generic_neuromotor_interface.networks import DiscreteGesturesArchitecture


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


# ------------------------------------------------------------
# Data/model timing and output semantics
# ------------------------------------------------------------

YOUR_CHANNELS = 32
META_CHANNELS = 16

YOUR_SAMPLING_RATE = 30_000
META_SAMPLING_RATE = 2_000

RAW_INPUT_SAMPLES = 30_000       # one second at 30 kHz
META_INPUT_SAMPLES = 2_000       # one second at 2 kHz
UTAH_TO_META_DOWNSAMPLE = YOUR_SAMPLING_RATE // META_SAMPLING_RATE

if YOUR_SAMPLING_RATE % META_SAMPLING_RATE != 0:
    raise ValueError("Utah-to-Meta sampling-rate ratio must be an integer.")

NUM_META_CLASSES = 9
AO_KEPT_CLASSES = 5

AO_CLASS_NAMES = [
    "thumb left",
    "thumb right",
    "thumb up",
    "thumb down",
    "thumb press",
]

# Meta output order:
# 0 index press
# 1 index release
# 2 middle press
# 3 middle release
# 4 thumb tap
# 5 thumb swipe left
# 6 thumb swipe right
# 7 thumb swipe up
# 8 thumb swipe down
UTAH_TO_META_OUTPUTS = [5, 6, 7, 8, 4]

if len(UTAH_TO_META_OUTPUTS) != AO_KEPT_CLASSES:
    raise ValueError("UTAH_TO_META_OUTPUTS must contain five indices.")

if len(set(UTAH_TO_META_OUTPUTS)) != AO_KEPT_CLASSES:
    raise ValueError("UTAH_TO_META_OUTPUTS contains duplicate indices.")

if not all(0 <= index < NUM_META_CLASSES for index in UTAH_TO_META_OUTPUTS):
    raise ValueError("UTAH_TO_META_OUTPUTS contains an invalid Meta output index.")


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

BATCH_SIZE = 16
AO_EPOCHS = 200

AO_LR = 0.01
AO_WEIGHT_DECAY = 1e-3
AO_GRAD_CLIP_NORM = 1.0

AO_WARMUP_EPOCHS = 5
AO_DECAY_EPOCH = 26
AO_DECAY_FACTOR = 0.5

# The saved Utah dataset is expected to already contain the previously chosen
# 100 ms forward label shift. This notebook does NOT shift labels a second time.
UTAH_LABELS_ALREADY_SHIFTED_100_MS = True

EXPECTED_SPLIT_SIZES = {
    "train": 80,
    "val": 10,
    "test": 10,
}


# ------------------------------------------------------------
# Preprocessing configuration
# ------------------------------------------------------------

HIGH_PASS_HZ = 40.0
HIGH_PASS_ORDER = 4

# Utah hardware units are dataset-specific. Change only if independently
# justified for this dataset.
UTAH_INPUT_SCALE = 1.0

# Official DiscreteGesturesArchitecture returns logits, not probabilities.
META_OUTPUT_IS_PROBABILITY = False


# ------------------------------------------------------------
# Adapter configuration
# ------------------------------------------------------------

ADAPTER_HIDDEN_CHANNELS = 48
ADAPTER_GROUPS = 8
ADAPTER_KERNEL_SIZE = 5
INITIAL_RESIDUAL_SCALE = 0.10
INITIAL_OUTPUT_GAIN = 0.25

# Meta-style local channel rotations. The same learnable adapter is applied to
# every view, and the resulting 16-channel representations are mean-pooled.
ROTATION_OFFSETS = (-1, 0, 1)
ROTATION_CHANNEL_GROUP_SIZE = 32

if not ROTATION_OFFSETS or 0 not in ROTATION_OFFSETS:
    raise ValueError("ROTATION_OFFSETS must be non-empty and contain zero.")
if len(set(ROTATION_OFFSETS)) != len(ROTATION_OFFSETS):
    raise ValueError("ROTATION_OFFSETS must not contain duplicates.")
if YOUR_CHANNELS % ROTATION_CHANNEL_GROUP_SIZE != 0:
    raise ValueError(
        "ROTATION_CHANNEL_GROUP_SIZE must evenly divide YOUR_CHANNELS."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------
# Signal preprocessing helpers
# ------------------------------------------------------------

def make_highpass_sos(sample_rate, cutoff_hz=40.0, order=4):
    if not 0 < cutoff_hz < sample_rate / 2:
        raise ValueError(
            f"High-pass cutoff {cutoff_hz} Hz is invalid for "
            f"sample rate {sample_rate} Hz."
        )

    return butter(
        order,
        cutoff_hz,
        btype="highpass",
        fs=sample_rate,
        output="sos",
    )


META_HIGHPASS_SOS = make_highpass_sos(
    META_SAMPLING_RATE,
    HIGH_PASS_HZ,
    HIGH_PASS_ORDER,
)


def highpass_numpy_channels_time(x_ct, sos=META_HIGHPASS_SOS):
    """
    Apply a zero-phase 40 Hz high-pass to [channels,time].
    """
    x = np.asarray(x_ct, dtype=np.float32)

    if x.ndim != 2:
        raise ValueError(f"Expected [channels,time], got {x.shape}")

    if x.shape[1] < 32:
        raise ValueError(
            f"Signal is too short for stable high-pass filtering: {x.shape}"
        )

    return sosfiltfilt(sos, x, axis=1).astype(np.float32, copy=False)


def resample_utah_30k_to_2k(x_ct):
    """
    Anti-aliased rational resampling:
        [32,30000] at 30 kHz -> [32,2000] at 2 kHz.
    """
    x = np.asarray(x_ct, dtype=np.float32)

    if x.ndim != 2:
        raise ValueError(f"Expected [channels,time], got {x.shape}")

    if x.shape[0] != YOUR_CHANNELS:
        raise ValueError(
            f"Expected {YOUR_CHANNELS} Utah channels, got {x.shape}"
        )

    if x.shape[1] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected exactly {RAW_INPUT_SAMPLES} Utah samples, got {x.shape}"
        )

    y = resample_poly(
        x,
        up=1,
        down=UTAH_TO_META_DOWNSAMPLE,
        axis=1,
    )

    if y.shape != (YOUR_CHANNELS, META_INPUT_SAMPLES):
        raise RuntimeError(
            "Unexpected anti-aliased resampling shape: "
            f"expected {(YOUR_CHANNELS, META_INPUT_SAMPLES)}, got {y.shape}"
        )

    return y.astype(np.float32, copy=False)


def preprocess_utah_trial(x_ct):
    """
    Utah:
        30 kHz -> anti-aliased 2 kHz -> 40 Hz high-pass -> optional Utah scale.
    """
    y = resample_utah_30k_to_2k(x_ct)
    y = highpass_numpy_channels_time(y)
    y = y * float(UTAH_INPUT_SCALE)
    return y.astype(np.float32, copy=False)


# ------------------------------------------------------------
# Load and freeze the official Meta model
# ------------------------------------------------------------

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Could not find Meta checkpoint:\n{CKPT_PATH}")

meta_model = DiscreteGesturesArchitecture(output_channels=NUM_META_CLASSES)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

if "state_dict" not in ckpt:
    raise KeyError("Checkpoint is missing 'state_dict'.")

state = ckpt["state_dict"]

network_state = {
    key.replace("network.", "", 1): value
    for key, value in state.items()
    if key.startswith("network.")
}

if not network_state:
    network_state = state

meta_model.load_state_dict(network_state, strict=True)

for parameter in meta_model.parameters():
    parameter.requires_grad = False

meta_model.eval()

# Verify the imported official architecture.
if not hasattr(meta_model, "compression"):
    raise RuntimeError("Meta model has no compression module.")

compression_range = float(getattr(meta_model.compression, "range", np.nan))
compression_midpoint = float(getattr(meta_model.compression, "midpoint", np.nan))

if not np.isclose(compression_range, 64.0):
    raise RuntimeError(
        f"Expected Meta compression range=64, got {compression_range}"
    )

if not np.isclose(compression_midpoint, 32.0):
    raise RuntimeError(
        f"Expected Meta compression midpoint=32, got {compression_midpoint}"
    )

META_LEFT_CONTEXT = int(meta_model.left_context)
META_OUTPUT_STRIDE = int(meta_model.stride)
EXPECTED_META_OUTPUT_SAMPLES = len(
    range(META_LEFT_CONTEXT, META_INPUT_SAMPLES, META_OUTPUT_STRIDE)
)


# ------------------------------------------------------------
# Rotation-robust regularized 32-channel -> 16-channel TCN adapter
# ------------------------------------------------------------

def make_group_norm(num_channels, requested_groups=8):
    groups = min(requested_groups, num_channels)

    while groups > 1 and num_channels % groups != 0:
        groups -= 1

    return nn.GroupNorm(groups, num_channels)


def make_wn_conv1d(
    in_channels,
    out_channels,
    kernel_size,
    padding=0,
    dilation=1,
    groups=1,
    bias=True,
):
    layer = nn.Conv1d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        padding=padding,
        dilation=dilation,
        groups=groups,
        bias=bias,
    )
    return weight_norm(layer)


class DepthwiseSeparableTemporalConv(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=5,
        dilation=1,
        norm_groups=8,
    ):
        super().__init__()

        padding = dilation * (kernel_size - 1) // 2

        self.depthwise = make_wn_conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
            groups=channels,
            bias=False,
        )

        self.pointwise = make_wn_conv1d(
            channels,
            channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = make_group_norm(channels, norm_groups)
        self.activation = nn.SiLU()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        return self.activation(x)


class RegularizedTCNBlock(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=5,
        dilation=1,
        norm_groups=8,
        initial_residual_scale=0.10,
    ):
        super().__init__()

        self.temporal_1 = DepthwiseSeparableTemporalConv(
            channels,
            kernel_size,
            dilation,
            norm_groups,
        )
        self.temporal_2 = DepthwiseSeparableTemporalConv(
            channels,
            kernel_size,
            dilation,
            norm_groups,
        )

        self.residual_scale = nn.Parameter(
            torch.tensor(float(initial_residual_scale))
        )

    def forward(self, x):
        residual = x
        out = self.temporal_1(x)
        out = self.temporal_2(out)
        return residual + self.residual_scale * out


def rotate_channels_within_groups(x, offset, group_size):
    """
    Apply one circular channel offset independently within each configured
    physical channel group. Input and output both have shape [B,C,T].
    """
    if x.ndim != 3:
        raise ValueError(f"Expected [B,C,T], got {tuple(x.shape)}")
    if x.shape[1] % group_size != 0:
        raise ValueError(
            f"Channel count {x.shape[1]} is not divisible by group size {group_size}."
        )
    batch, channels, time = x.shape
    groups = channels // group_size
    grouped = x.reshape(batch, groups, group_size, time)
    rotated = torch.roll(grouped, shifts=int(offset), dims=2)
    return rotated.reshape(batch, channels, time)


class AdapterToMetaInput(nn.Module):
    """
    [B,32,2000] -> [B,16,2000].

    The adapter changes the channel representation without changing physical
    duration or sampling rate. One shared nonlinear adapter processes each
    configured circular channel offset, then mean-pools the resulting learned
    representations. Sharing weights prevents the rotations from becoming
    separate independently fitted branches.
    """
    def __init__(
        self,
        input_channels=32,
        output_channels=16,
        hidden_channels=48,
    ):
        super().__init__()

        self.input_projection = nn.Sequential(
            make_wn_conv1d(
                input_channels,
                hidden_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(hidden_channels, ADAPTER_GROUPS),
            nn.SiLU(),
        )

        self.temporal_stack = nn.Sequential(
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=1,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=2,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=4,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=8,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
        )

        self.output_projection = make_wn_conv1d(
            hidden_channels,
            output_channels,
            kernel_size=1,
            bias=True,
        )

        self.output_gain = nn.Parameter(
            torch.tensor(float(INITIAL_OUTPUT_GAIN))
        )

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(f"Expected [B,32,T], got {tuple(x.shape)}")

        if x.shape[1] != YOUR_CHANNELS:
            raise ValueError(
                f"Expected {YOUR_CHANNELS} Utah channels, got {tuple(x.shape)}"
            )

        if x.shape[-1] != META_INPUT_SAMPLES:
            raise ValueError(
                f"Expected one second at 2 kHz (T={META_INPUT_SAMPLES}), "
                f"got T={x.shape[-1]}"
            )

        rotated_views = torch.stack(
            [
                rotate_channels_within_groups(
                    x,
                    offset=offset,
                    group_size=ROTATION_CHANNEL_GROUP_SIZE,
                )
                for offset in ROTATION_OFFSETS
            ],
            dim=1,
        )

        batch, rotations, channels, time = rotated_views.shape
        shared_input = rotated_views.reshape(batch * rotations, channels, time)
        shared_output = self.input_projection(shared_input)
        shared_output = self.temporal_stack(shared_output)
        shared_output = self.output_projection(shared_output)

        shared_output = shared_output.reshape(
            batch,
            rotations,
            shared_output.shape[1],
            shared_output.shape[2],
        )
        pooled_output = shared_output.mean(dim=1)
        return self.output_gain * pooled_output


class FrozenMetaWithAdapter(nn.Module):
    def __init__(self, frozen_meta_model):
        super().__init__()

        self.adapter = AdapterToMetaInput(
            input_channels=YOUR_CHANNELS,
            output_channels=META_CHANNELS,
            hidden_channels=ADAPTER_HIDDEN_CHANNELS,
        )

        self.meta_model = frozen_meta_model

        for parameter in self.meta_model.parameters():
            parameter.requires_grad = False

        self.meta_model.eval()

    def train(self, mode=True):
        super().train(mode)
        self.meta_model.eval()
        return self

    def forward_with_adapter(self, x):
        adapter_output = self.adapter(x)
        raw_meta_output = self.meta_model(adapter_output)
        return raw_meta_output, adapter_output

    def forward(self, x):
        raw_meta_output, _ = self.forward_with_adapter(x)
        return raw_meta_output


def module_numeric_signature(module):
    """
    Lightweight frozen-model mutation check.
    """
    total_sum = 0.0
    total_sq_sum = 0.0
    total_count = 0

    with torch.no_grad():
        for tensor in module.state_dict().values():
            value = tensor.detach().double().cpu()
            total_sum += float(value.sum().item())
            total_sq_sum += float(value.square().sum().item())
            total_count += value.numel()

    return (total_count, total_sum, total_sq_sum)


model = FrozenMetaWithAdapter(meta_model).to(DEVICE)
initial_adapter_state = copy.deepcopy(model.adapter.state_dict())
initial_meta_signature = module_numeric_signature(model.meta_model)

total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

if any(parameter.requires_grad for parameter in model.meta_model.parameters()):
    raise RuntimeError("The Meta backbone is not fully frozen.")

print("Using device:", DEVICE)
print("Physical timing:")
print(
    f"  Utah raw: {RAW_INPUT_SAMPLES} samples @ "
    f"{YOUR_SAMPLING_RATE} Hz = 1.000 s"
)
print(
    f"  Meta input: {META_INPUT_SAMPLES} samples @ "
    f"{META_SAMPLING_RATE} Hz = 1.000 s"
)
print(
    f"  Meta output: {EXPECTED_META_OUTPUT_SAMPLES} bins using "
    f"left_context={META_LEFT_CONTEXT}, stride={META_OUTPUT_STRIDE}"
)
print("Output mapping:", UTAH_TO_META_OUTPUTS)
print("Adapter rotation offsets:", ROTATION_OFFSETS)
print("Rotation channel group size:", ROTATION_CHANNEL_GROUP_SIZE)
print("Shared adapter views per trial:", len(ROTATION_OFFSETS))
print(
    "Meta internal compression:",
    f"{compression_range:g}*x/({compression_midpoint:g}+|x|)",
)
print("Total parameters:", f"{total_params:,}")
print("Trainable adapter parameters:", f"{trainable_params:,}")
print("Frozen parameters:", f"{total_params - trainable_params:,}")


Using device: cpu
Physical timing:
  Utah raw: 30000 samples @ 30000 Hz = 1.000 s
  Meta input: 2000 samples @ 2000 Hz = 1.000 s
  Meta output: 198 bins using left_context=20, stride=10
Output mapping: [5, 6, 7, 8, 4]
Adapter rotation offsets: (-1, 0, 1)
Rotation channel group size: 32
Shared adapter views per trial: 3
Meta internal compression: 64*x/(32+|x|)
Total parameters: 6,507,326
Trainable adapter parameters: 24,373
Frozen parameters: 6,482,953


In [2]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Orientation helpers
# ------------------------------------------------------------

def force_emg_time_channels(x):
    """Return Utah EMG as [time, 32]."""
    x = torch.as_tensor(x).float()

    if x.ndim != 2:
        raise ValueError(f"Expected 2D EMG, got {tuple(x.shape)}")

    if x.shape[1] == YOUR_CHANNELS:
        return x.contiguous()

    if x.shape[0] == YOUR_CHANNELS:
        return x.transpose(0, 1).contiguous()

    raise ValueError(
        f"Could not identify {YOUR_CHANNELS}-channel EMG orientation: "
        f"{tuple(x.shape)}"
    )


def force_labels_time_classes(x, expected_time):
    """Return labels as [time, classes]."""
    x = torch.as_tensor(x).float()

    if x.ndim != 2:
        raise ValueError(f"Expected 2D labels, got {tuple(x.shape)}")

    candidates = []

    if x.shape[0] == expected_time and x.shape[1] >= AO_KEPT_CLASSES:
        candidates.append(x.contiguous())

    if x.shape[1] == expected_time and x.shape[0] >= AO_KEPT_CLASSES:
        candidates.append(x.transpose(0, 1).contiguous())

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        raise ValueError(
            f"Ambiguous label orientation for shape {tuple(x.shape)} "
            f"and expected_time={expected_time}."
        )

    # Fallback when EMG and labels differ slightly in length.
    if x.shape[0] > x.shape[1] and x.shape[1] >= AO_KEPT_CLASSES:
        return x.contiguous()

    if x.shape[1] > x.shape[0] and x.shape[0] >= AO_KEPT_CLASSES:
        return x.transpose(0, 1).contiguous()

    raise ValueError(
        f"Could not identify label orientation: {tuple(x.shape)}"
    )


# ------------------------------------------------------------
# Trial-local active interval and one-second extraction
# ------------------------------------------------------------

def get_common_trial_arrays(trial):
    """
    Return aligned trial-local arrays:
        emg_tc:    [T,32]
        labels_tc: [T,C]
        valid_t:   [T]
    """
    if "ns5_vector" not in trial:
        raise KeyError("Trial is missing 'ns5_vector'.")

    if "trainKin" not in trial:
        raise KeyError("Trial is missing 'trainKin'.")

    emg_tc = force_emg_time_channels(trial["ns5_vector"])
    labels_tc = force_labels_time_classes(
        trial["trainKin"],
        expected_time=emg_tc.shape[0],
    )

    if "valid_mask" in trial:
        valid_t = torch.as_tensor(
            trial["valid_mask"]
        ).float().reshape(-1)
    else:
        valid_t = torch.ones(
            emg_tc.shape[0],
            dtype=torch.float32,
        )

    common_length = min(
        emg_tc.shape[0],
        labels_tc.shape[0],
        valid_t.shape[0],
    )

    if common_length <= 0:
        raise ValueError("Trial has no common EMG/label/mask samples.")

    return (
        emg_tc[:common_length].contiguous(),
        labels_tc[:common_length].contiguous(),
        valid_t[:common_length].contiguous(),
    )


def find_active_valid_interval(labels_tc, valid_t):
    """
    Find the first and last trial-local samples where:
        any Utah gesture label is active
        AND
        valid_mask == 1

    Returns None when the final saved trial has no usable active label.
    """
    active_t = (
        labels_tc[:, :AO_KEPT_CLASSES].amax(dim=1) > 0.5
    )
    valid_bool = valid_t > 0.5
    active_valid_t = active_t & valid_bool

    indices = torch.nonzero(
        active_valid_t,
        as_tuple=False,
    ).reshape(-1)

    if indices.numel() == 0:
        return None

    start = int(indices[0].item())
    end_exclusive = int(indices[-1].item()) + 1

    return start, end_exclusive


def extract_centered_time_window(
    x,
    center_index,
    target_samples,
    pad_value=0.0,
):
    """
    Extract one time-first window centered on `center_index`.

    If the full trial is at least target_samples long, the window is shifted
    at the boundaries so it contains only real samples.

    If the full trial is shorter than target_samples, it is padded while
    keeping the active interval center aligned to the middle of the output.

    Returns:
        output
        source_start
        source_end
        left_pad
        right_pad
    """
    if x.ndim < 1:
        raise ValueError("Input must have a time dimension.")

    total_samples = int(x.shape[0])
    target_samples = int(target_samples)
    center_index = int(center_index)

    if total_samples <= 0:
        raise ValueError("Cannot crop an empty trial.")

    if not 0 <= center_index < total_samples:
        raise ValueError(
            f"center_index={center_index} is outside trial length "
            f"{total_samples}."
        )

    if total_samples >= target_samples:
        source_start = center_index - target_samples // 2
        source_start = max(0, source_start)
        source_start = min(source_start, total_samples - target_samples)
        source_end = source_start + target_samples

        return (
            x[source_start:source_end].contiguous(),
            source_start,
            source_end,
            0,
            0,
        )

    # Short trial: preserve active-center alignment with explicit padding.
    desired_start = center_index - target_samples // 2
    desired_end = desired_start + target_samples

    source_start = max(0, desired_start)
    source_end = min(total_samples, desired_end)

    left_pad = source_start - desired_start
    copied = source_end - source_start
    right_pad = target_samples - left_pad - copied

    output_shape = (target_samples,) + tuple(x.shape[1:])
    output = torch.full(
        output_shape,
        float(pad_value),
        dtype=x.dtype,
    )

    destination_start = left_pad
    destination_end = destination_start + copied
    output[destination_start:destination_end] = x[source_start:source_end]

    return (
        output.contiguous(),
        source_start,
        source_end,
        left_pad,
        right_pad,
    )


def extract_one_second_active_centered_trial(trial):
    """
    Apply one identical trial-local crop/pad to EMG, labels, and valid mask.
    """
    emg_tc, labels_tc, valid_t = get_common_trial_arrays(trial)

    interval = find_active_valid_interval(labels_tc, valid_t)

    if interval is None:
        return None

    active_start, active_end = interval
    active_span = active_end - active_start

    if active_span > RAW_INPUT_SAMPLES:
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", -1))
        raise ValueError(
            f"G{gesture} trial {trial_num} has an active-valid interval "
            f"of {active_span} samples, longer than the one-second "
            f"window ({RAW_INPUT_SAMPLES})."
        )

    active_center = (active_start + active_end - 1) // 2

    emg_window, source_start, source_end, left_pad, right_pad = (
        extract_centered_time_window(
            emg_tc,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    labels_window, labels_source_start, labels_source_end, labels_left_pad, labels_right_pad = (
        extract_centered_time_window(
            labels_tc,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    valid_window, valid_source_start, valid_source_end, valid_left_pad, valid_right_pad = (
        extract_centered_time_window(
            valid_t,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    crop_metadata = {
        "original_length": int(emg_tc.shape[0]),
        "active_start": active_start,
        "active_end": active_end,
        "active_span": active_span,
        "active_center": active_center,
        "source_start": source_start,
        "source_end": source_end,
        "left_pad": left_pad,
        "right_pad": right_pad,
    }

    consistency_values = {
        (
            source_start,
            source_end,
            left_pad,
            right_pad,
        ),
        (
            labels_source_start,
            labels_source_end,
            labels_left_pad,
            labels_right_pad,
        ),
        (
            valid_source_start,
            valid_source_end,
            valid_left_pad,
            valid_right_pad,
        ),
    }

    if len(consistency_values) != 1:
        raise RuntimeError(
            "EMG, labels, and valid mask did not receive the same crop."
        )

    if emg_window.shape != (RAW_INPUT_SAMPLES, YOUR_CHANNELS):
        raise RuntimeError(
            f"Unexpected cropped EMG shape: {tuple(emg_window.shape)}"
        )

    if labels_window.shape[0] != RAW_INPUT_SAMPLES:
        raise RuntimeError(
            f"Unexpected cropped label shape: {tuple(labels_window.shape)}"
        )

    if valid_window.shape != (RAW_INPUT_SAMPLES,):
        raise RuntimeError(
            f"Unexpected cropped valid-mask shape: "
            f"{tuple(valid_window.shape)}"
        )

    return emg_window, labels_window, valid_window, crop_metadata


# ------------------------------------------------------------
# Exact 30 kHz -> 2 kHz label/mask reduction
# ------------------------------------------------------------

def downsample_binary_labels_30k_to_2k(labels_tc):
    """
    Exact 15:1 block max pooling.

    A positive label survives when any corresponding 30 kHz sample is positive.
    """
    if labels_tc.shape[0] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected {RAW_INPUT_SAMPLES} label samples, got "
            f"{tuple(labels_tc.shape)}"
        )

    labels_ct = labels_tc.transpose(0, 1).unsqueeze(0)

    pooled = F.max_pool1d(
        labels_ct,
        kernel_size=UTAH_TO_META_DOWNSAMPLE,
        stride=UTAH_TO_META_DOWNSAMPLE,
    )

    pooled_tc = pooled.squeeze(0).transpose(0, 1).contiguous()

    if pooled_tc.shape[0] != META_INPUT_SAMPLES:
        raise RuntimeError(
            f"Label downsampling produced {pooled_tc.shape[0]} samples."
        )

    return pooled_tc


def downsample_valid_mask_30k_to_2k(valid_t):
    """
    Exact conservative 15:1 validity reduction.

    A 2 kHz sample is valid only when every contributing 30 kHz sample is valid.
    """
    if valid_t.shape[0] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected {RAW_INPUT_SAMPLES} valid-mask samples, got "
            f"{tuple(valid_t.shape)}"
        )

    invalid = (1.0 - valid_t.clamp(0.0, 1.0)).reshape(1, 1, -1)

    pooled_invalid = F.max_pool1d(
        invalid,
        kernel_size=UTAH_TO_META_DOWNSAMPLE,
        stride=UTAH_TO_META_DOWNSAMPLE,
    )

    valid_2k = 1.0 - pooled_invalid.reshape(-1)

    if valid_2k.shape[0] != META_INPUT_SAMPLES:
        raise RuntimeError(
            f"Valid-mask downsampling produced {valid_2k.shape[0]} samples."
        )

    return valid_2k.float().contiguous()


# ------------------------------------------------------------


class GestureOneSecond2kHzDataset(Dataset):
    """
    One active-centered one-second trial per item.

    Returns:
        emg:        [32,2000]
        target:     [5,2000]
        valid_mask: [2000]
        gesture:    scalar 0..4
        trial_num:  scalar
    """
    def __init__(self, trials, split_name):
        self.trials = list(trials)
        self.split_name = str(split_name)

    def __len__(self):
        return len(self.trials)

    def __getitem__(self, index):
        trial = self.trials[index]
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", index))

        if not 0 <= gesture < AO_KEPT_CLASSES:
            raise ValueError(
                f"{self.split_name} trial {trial_num} has invalid "
                f"gesture={gesture}; expected 0..{AO_KEPT_CLASSES - 1}."
            )

        extracted = extract_one_second_active_centered_trial(trial)

        if extracted is None:
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} has no "
                "active-valid samples in its complete saved segment."
            )

        emg_tc, labels_tc, valid_t, _ = extracted

        emg_ct_np = emg_tc.transpose(0, 1).cpu().numpy()
        emg_2k_ct_np = preprocess_utah_trial(emg_ct_np)
        emg_2k_ct = torch.from_numpy(emg_2k_ct_np).float()

        target_2k_tc = downsample_binary_labels_30k_to_2k(
            labels_tc[:, :AO_KEPT_CLASSES]
        )
        target_2k_ct = target_2k_tc.transpose(0, 1).contiguous()

        valid_2k = downsample_valid_mask_30k_to_2k(valid_t)

        if not torch.isfinite(emg_2k_ct).all():
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} "
                "contains non-finite EMG."
            )

        if not torch.isfinite(target_2k_ct).all():
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} "
                "contains non-finite labels."
            )

        return (
            emg_2k_ct,
            target_2k_ct,
            valid_2k,
            torch.tensor(gesture, dtype=torch.long),
            torch.tensor(trial_num, dtype=torch.long),
        )


In [3]:
# ============================================================
# CELL 3 — EXACT META TARGET ALIGNMENT + MAPPED TASK BCE
# ============================================================

import torch
import torch.nn.functional as F


META_OUTPUT_INDEX_TENSOR = torch.tensor(
    UTAH_TO_META_OUTPUTS,
    dtype=torch.long,
    device=DEVICE,
)


def map_meta_outputs_to_utah(raw_meta_output):
    """
    Meta [B,9,T] -> Utah order [B,5,T]:
        left, right, up, down, thumb press/tap.
    """
    if raw_meta_output.ndim != 3:
        raise ValueError(
            f"Expected Meta output [B,9,T], got "
            f"{tuple(raw_meta_output.shape)}"
        )

    if raw_meta_output.shape[1] != NUM_META_CLASSES:
        raise ValueError(
            f"Expected {NUM_META_CLASSES} Meta outputs, got "
            f"{tuple(raw_meta_output.shape)}"
        )

    selected = raw_meta_output.index_select(
        dim=1,
        index=META_OUTPUT_INDEX_TENSOR,
    )

    if META_OUTPUT_IS_PROBABILITY:
        selected = torch.logit(
            selected.clamp(1e-5, 1.0 - 1e-5)
        )

    return selected


def align_target_and_mask_to_logits(target, valid_mask, output_length):
    """
    Match the official Meta training alignment exactly:

        target[..., left_context::stride]

    For the official discrete-gesture model this is:
        target[..., 20::10] -> 198 bins for a 2,000-sample input.
    """
    if target.ndim != 3:
        raise ValueError(
            f"Expected target [B,5,T], got {tuple(target.shape)}"
        )

    if valid_mask.ndim != 2:
        raise ValueError(
            f"Expected valid_mask [B,T], got {tuple(valid_mask.shape)}"
        )

    aligned_target = target[
        ...,
        META_LEFT_CONTEXT::META_OUTPUT_STRIDE,
    ]

    aligned_valid = valid_mask[
        ...,
        META_LEFT_CONTEXT::META_OUTPUT_STRIDE,
    ]

    if aligned_target.shape[-1] != output_length:
        raise RuntimeError(
            "Target alignment does not match Meta output length: "
            f"target={aligned_target.shape[-1]}, output={output_length}."
        )

    if aligned_valid.shape[-1] != output_length:
        raise RuntimeError(
            "Valid-mask alignment does not match Meta output length: "
            f"mask={aligned_valid.shape[-1]}, output={output_length}."
        )

    return aligned_target, aligned_valid


def ao_unpack_batch(batch):
    emg, target, valid_mask, gesture, trial_num = batch

    return (
        emg.to(DEVICE).float(),
        target.to(DEVICE).float(),
        valid_mask.to(DEVICE).float(),
        gesture.to(DEVICE).long(),
        trial_num.to(DEVICE).long(),
    )


def active_only_multilabel_bce(logits, target, valid_mask):
    """
    Independent five-output BCE over active valid Utah bins.

    Negative classes are still penalized within every active bin. Rest bins
    are intentionally excluded so that the small dataset is not dominated by
    all-zero time points.
    """
    active = target.max(dim=1).values > 0.5
    valid = valid_mask > 0.5
    keep = active & valid

    if int(keep.sum().item()) == 0:
        raise RuntimeError("Batch contains no active valid target bins.")

    per_class_loss = F.binary_cross_entropy_with_logits(
        logits,
        target,
        reduction="none",
    )

    keep_expanded = keep.unsqueeze(1).expand_as(per_class_loss)

    return per_class_loss[keep_expanded].mean()


print("Loss and alignment functions ready.")
print("  Exact target alignment:", f"{META_LEFT_CONTEXT}::{META_OUTPUT_STRIDE}")
print("  Mapped Meta outputs:", UTAH_TO_META_OUTPUTS)
print("  Task loss: active-only five-output BCEWithLogits")


Loss and alignment functions ready.
  Exact target alignment: 20::10
  Mapped Meta outputs: [5, 6, 7, 8, 4]
  Task loss: active-only five-output BCEWithLogits


In [4]:
def trial_predictions(logits, target, valid_mask, gestures):
    """
    Return one mean-logit prediction per complete trial.

    The mapped Meta outputs are trained with independent BCE, but the Utah
    task requires one gesture decision per trial. Therefore, logits are
    averaged across active-valid output bins and the largest mean logit is
    selected.
    """
    active = target.max(dim=1).values > 0.5
    valid = valid_mask > 0.5
    keep = active & valid

    results = []

    for batch_index in range(logits.shape[0]):
        trial_keep = keep[batch_index]

        if int(trial_keep.sum().item()) == 0:
            raise RuntimeError(
                f"Batch item {batch_index} has no active valid output bins."
            )

        true_class = int(gestures[batch_index].item())

        if not 0 <= true_class < AO_KEPT_CLASSES:
            raise ValueError(
                f"Invalid trial gesture id {true_class}."
            )

        mean_logits = logits[
            batch_index,
            :,
            trial_keep,
        ].mean(dim=1)

        prediction = int(mean_logits.argmax().item())

        # Softmax is used only to display normalized mutually exclusive scores.
        # Training remains independent-output BCEWithLogits.
        normalized_scores = torch.softmax(
            mean_logits,
            dim=0,
        ).detach().cpu()

        results.append({
            "batch_index": batch_index,
            "true": true_class,
            "prediction": prediction,
            "normalized_scores": normalized_scores,
            "active_bins": int(trial_keep.sum().item()),
        })

    return results



In [5]:

from pathlib import Path as _CrossDayPath
import csv as _cross_csv
import hashlib as _cross_hashlib
import json as _cross_json
import numpy as _cross_np
import torch as _cross_torch
from torch.utils.data import DataLoader as _CrossDataLoader
from scipy.stats import binomtest as _cross_binomtest
import matplotlib.pyplot as _cross_plt

_cross_dataset_path = _CrossDayPath(
    r"C:\Users\Micah\utah-neuro\generic_neuromotor_interface\Gesture_Trial_Dataset_Labeled.pt"
)
_cross_checkpoint_path = _CrossDayPath(
    r"C:\Users\Micah\utah-neuro\MATLAB_Jupyter\02_rotation_invariance\figures_rotation_invariant\best_rotation_invariant_meta_adapter.pt"
)
_cross_output_dir = _CrossDayPath(
    r"C:\Users\Micah\utah-neuro\MATLAB_Jupyter\02_rotation_invariance\figures_rotation_invariant\cross_day_original_dataset"
)
_cross_output_dir.mkdir(parents=True, exist_ok=True)


def _cross_state_sha256(module):
    digest = _cross_hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        digest.update(name.encode("utf-8"))
        contiguous = value.detach().cpu().contiguous()
        digest.update(str(contiguous.dtype).encode("utf-8"))
        digest.update(_cross_np.asarray(contiguous.shape, dtype=_cross_np.int64).tobytes())
        digest.update(contiguous.numpy().tobytes())
    return digest.hexdigest()


_cross_checkpoint = _cross_torch.load(
    _cross_checkpoint_path,
    map_location="cpu",
    weights_only=False,
)
model.adapter.load_state_dict(_cross_checkpoint["adapter_state_dict"], strict=True)

if tuple(_cross_checkpoint["rotation_offsets"]) != tuple(ROTATION_OFFSETS):
    raise RuntimeError("Checkpoint rotation offsets do not match the model definition.")
if int(_cross_checkpoint["rotation_channel_group_size"]) != int(ROTATION_CHANNEL_GROUP_SIZE):
    raise RuntimeError("Checkpoint rotation group size does not match the model definition.")

model.to(DEVICE).eval()
for _cross_parameter in model.parameters():
    _cross_parameter.requires_grad_(False)

if any(parameter.requires_grad for parameter in model.parameters()):
    raise RuntimeError("Frozen cross-day model still has trainable parameters.")

_cross_hash_before = _cross_state_sha256(model)
_cross_saved = _cross_torch.load(
    _cross_dataset_path,
    map_location="cpu",
    weights_only=False,
)
if "all_trials" not in _cross_saved:
    raise KeyError("Original dataset is missing all_trials.")

_cross_all_trials = list(_cross_saved["all_trials"])
_cross_usable_trials = []
_cross_unusable = []

for _cross_index, _cross_trial in enumerate(_cross_all_trials):
    _cross_gesture = int(_cross_trial.get("gesture", -1))
    _cross_trial_num = int(_cross_trial.get("trial_num", _cross_index))
    try:
        _cross_extracted = extract_one_second_active_centered_trial(_cross_trial)
    except Exception as _cross_error:
        _cross_unusable.append({
            "index": _cross_index,
            "gesture": _cross_gesture,
            "trial_num": _cross_trial_num,
            "reason": f"{type(_cross_error).__name__}: {_cross_error}",
        })
        continue
    if _cross_extracted is None:
        _cross_unusable.append({
            "index": _cross_index,
            "gesture": _cross_gesture,
            "trial_num": _cross_trial_num,
            "reason": "no active-valid labeled samples",
        })
        continue
    _cross_usable_trials.append(_cross_trial)

_cross_dataset = GestureOneSecond2kHzDataset(
    _cross_usable_trials,
    "cross_day_original",
)
_cross_loader = _CrossDataLoader(
    _cross_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

_cross_confusion = _cross_torch.zeros(
    AO_KEPT_CLASSES,
    AO_KEPT_CLASSES,
    dtype=_cross_torch.long,
)
_cross_rows = []
_cross_correct_trials = 0
_cross_total_trials = 0
_cross_correct_bins = 0
_cross_total_bins = 0
_cross_loss_sum = 0.0
_cross_batches = 0

with _cross_torch.inference_mode():
    for _cross_batch in _cross_loader:
        _cross_emg, _cross_target, _cross_valid, _cross_gestures, _cross_trial_nums = _cross_batch
        _cross_emg = _cross_emg.to(DEVICE)
        _cross_target = _cross_target.to(DEVICE)
        _cross_valid = _cross_valid.to(DEVICE)
        _cross_gestures = _cross_gestures.to(DEVICE)

        _cross_raw = model(_cross_emg)
        _cross_logits = map_meta_outputs_to_utah(_cross_raw)
        _cross_target_aligned, _cross_valid_aligned = align_target_and_mask_to_logits(
            _cross_target,
            _cross_valid,
            _cross_logits.shape[-1],
        )
        _cross_loss = active_only_multilabel_bce(
            _cross_logits,
            _cross_target_aligned,
            _cross_valid_aligned,
        )
        _cross_loss_sum += float(_cross_loss)
        _cross_batches += 1

        _cross_results = trial_predictions(
            _cross_logits,
            _cross_target_aligned,
            _cross_valid_aligned,
            _cross_gestures,
        )

        _cross_active = _cross_target_aligned.max(dim=1).values > 0.5
        _cross_keep = _cross_active & (_cross_valid_aligned > 0.5)

        for _cross_result in _cross_results:
            _cross_b = int(_cross_result["batch_index"])
            _cross_true = int(_cross_result["true"])
            _cross_pred = int(_cross_result["prediction"])
            _cross_scores = _cross_result["normalized_scores"].tolist()
            _cross_trial_num = int(_cross_trial_nums[_cross_b].item())
            _cross_is_correct = int(_cross_true == _cross_pred)

            _cross_correct_trials += _cross_is_correct
            _cross_total_trials += 1
            _cross_confusion[_cross_true, _cross_pred] += 1

            _cross_trial_keep = _cross_keep[_cross_b]
            _cross_true_bins = _cross_target_aligned[_cross_b, :, _cross_trial_keep].argmax(dim=0)
            _cross_pred_bins = _cross_logits[_cross_b, :, _cross_trial_keep].argmax(dim=0)
            _cross_bin_correct = int((_cross_true_bins == _cross_pred_bins).sum().item())
            _cross_bin_total = int(_cross_trial_keep.sum().item())
            _cross_correct_bins += _cross_bin_correct
            _cross_total_bins += _cross_bin_total

            _cross_rows.append({
                "gesture": _cross_true,
                "gesture_name": AO_CLASS_NAMES[_cross_true],
                "trial_num": _cross_trial_num,
                "prediction": _cross_pred,
                "prediction_name": AO_CLASS_NAMES[_cross_pred],
                "correct": _cross_is_correct,
                "active_bins": _cross_bin_total,
                "correct_active_bins": _cross_bin_correct,
                **{f"score_{index}_{AO_CLASS_NAMES[index]}": float(score) for index, score in enumerate(_cross_scores)},
            })

_cross_hash_after = _cross_state_sha256(model)
if _cross_hash_before != _cross_hash_after:
    raise RuntimeError("Frozen model weights changed during cross-day inference.")

_cross_accuracy = _cross_correct_trials / max(1, _cross_total_trials)
_cross_bin_accuracy = _cross_correct_bins / max(1, _cross_total_bins)
_cross_chance = 1.0 / AO_KEPT_CLASSES
_cross_test = _cross_binomtest(
    _cross_correct_trials,
    _cross_total_trials,
    p=_cross_chance,
    alternative="greater",
)
_cross_ci = _cross_binomtest(
    _cross_correct_trials,
    _cross_total_trials,
).proportion_ci(confidence_level=0.95, method="exact")

_cross_per_class = {}
for _cross_class in range(AO_KEPT_CLASSES):
    _cross_count = int(_cross_confusion[_cross_class].sum().item())
    _cross_class_correct = int(_cross_confusion[_cross_class, _cross_class].item())
    _cross_per_class[AO_CLASS_NAMES[_cross_class]] = {
        "trials": _cross_count,
        "correct": _cross_class_correct,
        "recall": _cross_class_correct / max(1, _cross_count),
    }

_cross_summary = {
    "protocol": (
        "Frozen offline cross-day classification. Ground-truth labels are used "
        "only for active-centered window extraction, active-bin selection, and scoring."
    ),
    "training_dataset": str(_cross_checkpoint["training_dataset"]),
    "evaluation_dataset": str(_cross_dataset_path),
    "checkpoint": str(_cross_checkpoint_path),
    "checkpoint_best_epoch": int(_cross_checkpoint["best_epoch"]),
    "checkpoint_best_validation_accuracy": float(_cross_checkpoint["best_validation_accuracy"]),
    "rotation_offsets": list(_cross_checkpoint["rotation_offsets"]),
    "rotation_channel_group_size": int(_cross_checkpoint["rotation_channel_group_size"]),
    "model_sha256_before": _cross_hash_before,
    "model_sha256_after": _cross_hash_after,
    "weights_unchanged": _cross_hash_before == _cross_hash_after,
    "dataset_trials": len(_cross_all_trials),
    "scored_trials": _cross_total_trials,
    "unusable_trials": len(_cross_unusable),
    "correct_trials": _cross_correct_trials,
    "complete_trial_accuracy": _cross_accuracy,
    "chance_accuracy": _cross_chance,
    "above_chance_binomial_p_one_sided": float(_cross_test.pvalue),
    "accuracy_95_percent_exact_ci": [float(_cross_ci.low), float(_cross_ci.high)],
    "active_bin_accuracy": _cross_bin_accuracy,
    "active_bins": _cross_total_bins,
    "mean_active_bin_bce": _cross_loss_sum / max(1, _cross_batches),
    "confusion_true_rows_predicted_columns": _cross_confusion.tolist(),
    "per_class": _cross_per_class,
    "unusable_details": _cross_unusable,
}

with (_cross_output_dir / "summary.json").open("w", encoding="utf-8") as _cross_handle:
    _cross_json.dump(_cross_summary, _cross_handle, indent=2)

with (_cross_output_dir / "trial_predictions.csv").open("w", newline="", encoding="utf-8") as _cross_handle:
    _cross_writer = _cross_csv.DictWriter(_cross_handle, fieldnames=list(_cross_rows[0].keys()))
    _cross_writer.writeheader()
    _cross_writer.writerows(_cross_rows)

_cross_np.savetxt(
    _cross_output_dir / "confusion_matrix.csv",
    _cross_confusion.numpy(),
    delimiter=",",
    fmt="%d",
)

_cross_figure, _cross_axis = _cross_plt.subplots(figsize=(7, 6))
_cross_image = _cross_axis.imshow(_cross_confusion.numpy(), cmap="Blues")
_cross_figure.colorbar(_cross_image, ax=_cross_axis)
_cross_axis.set_xticks(range(AO_KEPT_CLASSES), AO_CLASS_NAMES, rotation=35, ha="right")
_cross_axis.set_yticks(range(AO_KEPT_CLASSES), AO_CLASS_NAMES)
_cross_axis.set_xlabel("Predicted gesture")
_cross_axis.set_ylabel("True gesture")
_cross_axis.set_title(
    f"Frozen rotation-invariant cross-day evaluation\n"
    f"accuracy={_cross_accuracy:.1%}, chance={_cross_chance:.1%}, p={_cross_test.pvalue:.3g}"
)
for _cross_i in range(AO_KEPT_CLASSES):
    for _cross_j in range(AO_KEPT_CLASSES):
        _cross_axis.text(_cross_j, _cross_i, int(_cross_confusion[_cross_i, _cross_j]), ha="center", va="center")
_cross_figure.tight_layout()
for _cross_extension in ("png", "pdf", "svg"):
    _cross_figure.savefig(
        _cross_output_dir / f"confusion_matrix.{_cross_extension}",
        dpi=300 if _cross_extension == "png" else None,
        bbox_inches="tight",
    )
_cross_plt.close(_cross_figure)

print("CROSS_DAY_FROZEN_EVALUATION_COMPLETE")
print(_cross_json.dumps(_cross_summary, indent=2))
print("Saved:", _cross_output_dir.resolve())

CROSS_DAY_FROZEN_EVALUATION_COMPLETE
{
  "protocol": "Frozen offline cross-day classification. Ground-truth labels are used only for active-centered window extraction, active-bin selection, and scoring.",
  "training_dataset": "C:\\Users\\Micah\\utah-neuro\\generic_neuromotor_interface\\New_Gesture_Trial_Dataset_Labeled (2).pt",
  "evaluation_dataset": "C:\\Users\\Micah\\utah-neuro\\generic_neuromotor_interface\\Gesture_Trial_Dataset_Labeled.pt",
  "checkpoint": "C:\\Users\\Micah\\utah-neuro\\MATLAB_Jupyter\\figures_rotation_invariant\\best_rotation_invariant_meta_adapter.pt",
  "checkpoint_best_epoch": 191,
  "checkpoint_best_validation_accuracy": 1.0,
  "rotation_offsets": [
    -1,
    0,
    1
  ],
  "rotation_channel_group_size": 32,
  "model_sha256_before": "636ca3969bb538c7328150e6fb6c5d4ba83afdfc9fd82b8cea6734ef510e46c8",
  "model_sha256_after": "636ca3969bb538c7328150e6fb6c5d4ba83afdfc9fd82b8cea6734ef510e46c8",
  "weights_unchanged": true,
  "dataset_trials": 53,
  "scored_tri